<a href="https://colab.research.google.com/github/ajrodloptfg06/TFG-DSM-DeepLearning/blob/main/TFG_DsmV6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Mon Apr  6 10:16:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip -q install --upgrade pip
!pip -q install timm einops albumentations torchmetrics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 42.1 MB/s eta 0:00:00


In [ ]:
import os, random, time, math
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
from einops import rearrange

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    #6seg

Device: cuda
GPU: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/TFG_DSM"
CKPT_DIR = os.path.join(BASE, "checkpoints")
RES_DIR  = os.path.join(BASE, "results")
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RES_DIR, exist_ok=True)

print("CKPT_DIR:", CKPT_DIR)
print("RES_DIR:", RES_DIR)

Mounted at /content/drive
CKPT_DIR: /content/drive/MyDrive/TFG_DSM/checkpoints
RES_DIR: /content/drive/MyDrive/TFG_DSM/results


In [ ]:
CFG = {
    "in_channels": 4,
    "batch_size": 16,
    "num_workers": 2,
    "lr": 1e-4,
    "epochs": 50,
    "weight_decay": 1e-4,
    "ckpt_dir": CKPT_DIR,
    "results_dir": RES_DIR,
}
CFG

{'in_channels': 4,
 'batch_size': 16,
 'num_workers': 2,
 'lr': 0.0001,
 'epochs': 50,
 'weight_decay': 0.0001,
 'ckpt_dir': '/content/drive/MyDrive/TFG_DSM/checkpoints',
 'results_dir': '/content/drive/MyDrive/TFG_DSM/results'}

In [ ]:
import numpy as np

BASE = "/content/drive/MyDrive"

x_train = np.load(f"{BASE}/xtrain_nyc-002.npy")
y_train = np.load(f"{BASE}/ytrain_nyc.npy")
x_test  = np.load(f"{BASE}/xtest_nyc.npy")
y_test  = np.load(f"{BASE}/ytest_nyc.npy")

print("x_train:", x_train.shape, x_train.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("x_test:", x_test.shape)
print("y_test:", y_test.shape)

print("x_train min/max:", x_train.min(), x_train.max())
print("y_train min/max:", y_train.min(), y_train.max())


## 1 min

x_train: (6000, 128, 128, 4) float64
y_train: (6000, 128, 128, 1) float64
x_test: (330, 128, 128, 4)
y_test: (330, 128, 128, 1)
x_train min/max: -3.7899852752685548 16.885
y_train min/max: 0.0 405.8377990722656


In [ ]:
import numpy as np
import torch

# Convertimos a float32 (reduce memoria y acelera)
x_train_t = torch.from_numpy(x_train.astype(np.float32)).permute(0, 3, 1, 2)  # NHWC -> NCHW
y_train_t = torch.from_numpy(y_train.astype(np.float32)).permute(0, 3, 1, 2)  # (N,H,W,1) -> (N,1,H,W)

x_test_t  = torch.from_numpy(x_test.astype(np.float32)).permute(0, 3, 1, 2)
y_test_t  = torch.from_numpy(y_test.astype(np.float32)).permute(0, 3, 1, 2)

print("x_train_t:", x_train_t.shape, x_train_t.dtype)
print("y_train_t:", y_train_t.shape, y_train_t.dtype)
print("x_test_t:", x_test_t.shape)
print("y_test_t:", y_test_t.shape)

x_train_t: torch.Size([6000, 4, 128, 128]) torch.float32
y_train_t: torch.Size([6000, 1, 128, 128]) torch.float32
x_test_t: torch.Size([330, 4, 128, 128])
y_test_t: torch.Size([330, 1, 128, 128])


In [ ]:
from torch.utils.data import TensorDataset, DataLoader, random_split

dataset = TensorDataset(x_train_t, y_train_t)

val_ratio = 0.1
val_size = int(len(dataset) * val_ratio)
train_size = len(dataset) - val_size

train_ds, val_ds = random_split(dataset, [train_size, val_size],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

test_loader  = DataLoader(TensorDataset(x_test_t, y_test_t), batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print("train batches:", len(train_loader))
print("val batches:", len(val_loader))
print("test batches:", len(test_loader))

train batches: 338
val batches: 38
test batches: 21


In [ ]:
import torch

@torch.no_grad()
def mae(pred, target):
    return torch.mean(torch.abs(pred - target))

@torch.no_grad()
def rmse(pred, target):
    return torch.sqrt(torch.mean((pred - target) ** 2))

@torch.no_grad()
def r2_score(pred, target):
    # R2 = 1 - SS_res/SS_tot
    y = target
    yhat = pred
    y_mean = torch.mean(y)
    ss_res = torch.sum((y - yhat) ** 2)
    ss_tot = torch.sum((y - y_mean) ** 2).clamp_min(1e-6)
    return 1.0 - ss_res / ss_tot

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW



criterion = nn.SmoothL1Loss(beta=1.0)  # Huber

def train_one_epoch(model, loader, optimizer):
    model.train()
    total = 0.0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        pred = model(x)

        # asegurar mismo tamaño (por si un modelo devuelve otra resolución)
        if pred.shape[-2:] != y.shape[-2:]:
            pred = F.interpolate(pred, size=y.shape[-2:], mode="bilinear", align_corners=False)

        loss = criterion(pred, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total += loss.item()
    return total / max(1, len(loader))

@torch.no_grad()
def evaluate_global(model, loader):
    model.eval()

    abs_sum = 0.0
    sq_sum = 0.0
    n = 0

    preds_all = []
    targets_all = []

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        pred = model(x)

        if pred.shape[-2:] != y.shape[-2:]:
            pred = F.interpolate(pred, size=y.shape[-2:], mode="bilinear", align_corners=False)

        diff = pred - y

        abs_sum += torch.sum(torch.abs(diff)).item()
        sq_sum += torch.sum(diff ** 2).item()
        n += y.numel()

        preds_all.append(pred.flatten().cpu())
        targets_all.append(y.flatten().cpu())

    mae = abs_sum / n
    rmse = (sq_sum / n) ** 0.5

    preds_all = torch.cat(preds_all)
    targets_all = torch.cat(targets_all)

    y_mean = torch.mean(targets_all)
    ss_res = torch.sum((targets_all - preds_all) ** 2)
    ss_tot = torch.sum((targets_all - y_mean) ** 2).clamp_min(1e-6)
    r2 = 1.0 - (ss_res / ss_tot)

    return {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "R2": float(r2)
    }

In [ ]:
import os, time
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

def fit_model_resumable(model, train_loader, val_loader,
                        epochs=50, lr=1e-4, weight_decay=1e-4,
                        ckpt_path="checkpoint.pth"):
    model = model.to(device)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

    start_epoch = 1
    best_rmse = float("inf")
    history = []

    # Reanudar si existe checkpoint
    if os.path.exists(ckpt_path):
        print("Reanudando desde checkpoint:", ckpt_path)
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        start_epoch = ckpt["epoch"] + 1
        best_rmse = ckpt["best_rmse"]
        history = ckpt["history"]
        print("   -> continúa desde epoch", start_epoch)

    for epoch in range(start_epoch, epochs + 1):
        t0 = time.time()

        tr_loss = train_one_epoch(model, train_loader, optimizer)
        val_metrics = evaluate_global(model, val_loader)
        scheduler.step()

        lr_now = optimizer.param_groups[0]["lr"]
        dt = time.time() - t0

        row = {
            "epoch": epoch,
            "lr": lr_now,
            "train_loss": float(tr_loss),
            "val_MAE": float(val_metrics["MAE"]),
            "val_RMSE": float(val_metrics["RMSE"]),
            "val_R2": float(val_metrics["R2"]),
        }
        history.append(row)

        print(f"Epoch {epoch:03d} | lr={lr_now:.2e} | train_loss={tr_loss:.4f} | "
              f"val_RMSE={val_metrics['RMSE']:.4f} | val_MAE={val_metrics['MAE']:.4f} | val_R2={val_metrics['R2']:.4f} | {dt:.1f}s")

        if val_metrics["RMSE"] < best_rmse:
            best_rmse = val_metrics["RMSE"]

        # Guardar SIEMPRE (modelo + optimizer + scheduler + history)
        torch.save({
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "best_rmse": best_rmse,
            "history": history
        }, ckpt_path)

    return history, best_rmse

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

class ConvBNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, s, p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU()
        )
    def forward(self, x):
        return self.block(x)

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.conv1 = ConvBNAct(in_ch + skip_ch, out_ch)
        self.conv2 = ConvBNAct(out_ch, out_ch)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        return x

class SwinUNet(nn.Module):
    def __init__(self, in_ch=4, encoder_name="swin_tiny_patch4_window7_224", pretrained=True, img_size=128):
        super().__init__()

        self.encoder = timm.create_model(
            encoder_name,
            pretrained=pretrained,
            features_only=True,
            in_chans=in_ch,
            img_size=img_size,  # <-- acepta 128x128
        )
        enc_chs = self.encoder.feature_info.channels()

        self.bottleneck = nn.Sequential(
            ConvBNAct(enc_chs[-1], 256),
            ConvBNAct(256, 256),
        )

        self.dec3 = DecoderBlock(256, enc_chs[-2], 256)
        self.dec2 = DecoderBlock(256, enc_chs[-3], 128)
        self.dec1 = DecoderBlock(128, enc_chs[-4], 64)

        self.head = nn.Sequential(
            ConvBNAct(64, 32),
            nn.Conv2d(32, 1, kernel_size=1)
        )

    def forward(self, x):
      x_in_h, x_in_w = x.shape[-2], x.shape[-1]

      feats = self.encoder(x)  # lista de features NHWC

    # Convertir NHWC -> NCHW
      feats = [f.permute(0, 3, 1, 2) for f in feats]

      f1, f2, f3, f4 = feats

      x = self.bottleneck(f4)
      x = self.dec3(x, f3)
      x = self.dec2(x, f2)
      x = self.dec1(x, f1)

      x = self.head(x)
      x = F.interpolate(x, size=(x_in_h, x_in_w), mode="bilinear", align_corners=False)
      return x

In [ ]:
model = SwinUNet(in_ch=4, pretrained=True).to(device)

x, y = next(iter(train_loader))
x = x.to(device)
with torch.no_grad():
    pred = model(x)

print("pred:", pred.shape, "target:", y.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/timm/layers/interpolate.py:47: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at /pytorch/aten/src/ATen/native/BucketizationUtils.h:32.)
  idx_right = torch.bucketize(x, p)
/usr/local/lib/python3.12/dist-packages/timm/layers/interpolate.py:65: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  numerator += self.values[as_s] * \


pred: torch.Size([16, 1, 128, 128]) target: torch.Size([16, 1, 128, 128])


In [ ]:

swin_ckpt = os.path.join(CKPT_DIR, "best_swin_unet.pth")

model_swin = SwinUNet(in_ch=4, pretrained=True, img_size=128).to(device)

history_swin, best_rmse_swin = fit_model_resumable(
    model_swin,
    train_loader,
    val_loader,
    epochs=50,
    lr=1e-4,
    weight_decay=1e-4,
    ckpt_path=swin_ckpt
)

best_rmse_swin

#45 min

Reanudando desde checkpoint: /content/drive/MyDrive/TFG_DSM/checkpoints/best_swin_unet.pth
   -> continúa desde epoch 51


6.228235415589913

In [ ]:
ckpt = torch.load(swin_ckpt, map_location=device)
model_swin.load_state_dict(ckpt["model"])

test_metrics_swin = evaluate_global(model_swin, test_loader)
test_metrics_swin

{'MAE': 4.95777078108354,
 'RMSE': 10.615046921041836,
 'R2': 0.30705952644348145}

In [ ]:
results = []

results.append({
    "model": "Swin-UNet",
    "best_val_RMSE": float(ckpt["best_rmse"]),
    "test_MAE": test_metrics_swin["MAE"],
    "test_RMSE": test_metrics_swin["RMSE"],
    "test_R2": test_metrics_swin["R2"],
})

results

[{'model': 'Swin-UNet',
  'best_val_RMSE': 6.228235415589913,
  'test_MAE': 4.95777078108354,
  'test_RMSE': 10.615046921041836,
  'test_R2': 0.30705952644348145}]

In [ ]:
import pandas as pd
import os

df_results = pd.DataFrame(results)
results_csv_path = os.path.join(RES_DIR, "results.csv")
df_results.to_csv(results_csv_path, index=False)

print("Resultados guardados en:", results_csv_path)
df_results

Resultados guardados en: /content/drive/MyDrive/TFG_DSM/results/results.csv


,model,best_val_RMSE,test_MAE,test_RMSE,test_R2
0,Swin-UNet,6.228235,4.957771,10.615047,0.30706


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class AttentionGate(nn.Module):
    """
    g = gating (del decoder)
    x = skip (del encoder)
    """
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, g):
        # x: skip, g: decoder feature (resolución similar o se interpola)
        if g.shape[-2:] != x.shape[-2:]:
            g = F.interpolate(g, size=x.shape[-2:], mode="bilinear", align_corners=False)

        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

In [ ]:
class AttentionUNet(nn.Module):
    def __init__(self, in_ch=4, base_ch=64):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(in_ch, base_ch)           # 128
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(base_ch, base_ch*2)       # 64
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = DoubleConv(base_ch*2, base_ch*4)     # 32
        self.pool3 = nn.MaxPool2d(2)
        self.enc4 = DoubleConv(base_ch*4, base_ch*8)     # 16
        self.pool4 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(base_ch*8, base_ch*16)  # 8

        # Decoder (upsample + attention + conv)
        self.up4 = nn.ConvTranspose2d(base_ch*16, base_ch*8, kernel_size=2, stride=2)
        self.att4 = AttentionGate(F_g=base_ch*8, F_l=base_ch*8, F_int=base_ch*4)
        self.dec4 = DoubleConv(base_ch*16, base_ch*8)

        self.up3 = nn.ConvTranspose2d(base_ch*8, base_ch*4, kernel_size=2, stride=2)
        self.att3 = AttentionGate(F_g=base_ch*4, F_l=base_ch*4, F_int=base_ch*2)
        self.dec3 = DoubleConv(base_ch*8, base_ch*4)

        self.up2 = nn.ConvTranspose2d(base_ch*4, base_ch*2, kernel_size=2, stride=2)
        self.att2 = AttentionGate(F_g=base_ch*2, F_l=base_ch*2, F_int=base_ch)
        self.dec2 = DoubleConv(base_ch*4, base_ch*2)

        self.up1 = nn.ConvTranspose2d(base_ch*2, base_ch, kernel_size=2, stride=2)
        self.att1 = AttentionGate(F_g=base_ch, F_l=base_ch, F_int=base_ch//2)
        self.dec1 = DoubleConv(base_ch*2, base_ch)

        # Head
        self.out = nn.Conv2d(base_ch, 1, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))

        b = self.bottleneck(self.pool4(e4))

        # Decoder + attention on skips
        d4 = self.up4(b)
        e4_att = self.att4(e4, d4)
        d4 = self.dec4(torch.cat([d4, e4_att], dim=1))

        d3 = self.up3(d4)
        e3_att = self.att3(e3, d3)
        d3 = self.dec3(torch.cat([d3, e3_att], dim=1))

        d2 = self.up2(d3)
        e2_att = self.att2(e2, d2)
        d2 = self.dec2(torch.cat([d2, e2_att], dim=1))

        d1 = self.up1(d2)
        e1_att = self.att1(e1, d1)
        d1 = self.dec1(torch.cat([d1, e1_att], dim=1))

        return self.out(d1)

In [ ]:
model_att = AttentionUNet(in_ch=4, base_ch=64).to(device)

x, y = next(iter(train_loader))
x = x.to(device)
with torch.no_grad():
    pred = model_att(x)

print("pred:", pred.shape, "target:", y.shape)

pred: torch.Size([16, 1, 128, 128]) target: torch.Size([16, 1, 128, 128])


In [ ]:
att_ckpt = os.path.join(CKPT_DIR, "best_attention_unet.pth")

history_att, best_rmse_att = fit_model_resumable(
    model_att,
    train_loader,
    val_loader,
    epochs=50,
    lr=1e-4,
    weight_decay=1e-4,
    ckpt_path=att_ckpt
)

ckpt = torch.load(att_ckpt, map_location=device)
model_att.load_state_dict(ckpt["model"])

test_metrics_att = evaluate_global(model_att, test_loader)
test_metrics_att

#1Hora o mas

Reanudando desde checkpoint: /content/drive/MyDrive/TFG_DSM/checkpoints/best_attention_unet.pth
   -> continúa desde epoch 29
Epoch 029 | lr=3.76e-05 | train_loss=1.2113 | val_RMSE=4.0244 | val_MAE=1.6122 | val_R2=0.8024 | 87.4s
Epoch 030 | lr=3.45e-05 | train_loss=1.1853 | val_RMSE=3.8833 | val_MAE=1.5970 | val_R2=0.8160 | 95.9s
Epoch 031 | lr=3.16e-05 | train_loss=1.1826 | val_RMSE=4.0308 | val_MAE=1.5896 | val_R2=0.8018 | 95.9s


In [ ]:
results.append({
    "model": "Attention-U-Net",
    "best_val_RMSE": float(ckpt["best_rmse"]),
    "test_MAE": test_metrics_att["MAE"],
    "test_RMSE": test_metrics_att["RMSE"],
    "test_R2": test_metrics_att["R2"],
})
results

[{'model': 'Swin-UNet',
  'best_val_RMSE': 5.965726858691165,
  'test_loss': 4.527479535057431,
  'test_MAE': 4.9765592983790805,
  'test_RMSE': 8.626991033554077,
  'test_R2': 0.27468970843723844},
 {'model': 'Attention-U-Net',
  'best_val_RMSE': 3.2598106767001904,
  'test_loss': 3.647054677917844,
  'test_MAE': 4.061401821318126,
  'test_RMSE': 7.3578837144942515,
  'test_R2': 0.47026980774743216}]

In [ ]:


df_results = pd.DataFrame(results)
results_csv_path = os.path.join(RES_DIR, "results.csv")
df_results.to_csv(results_csv_path, index=False)

print("Resultados guardados en:", results_csv_path)
df_results

Resultados guardados en: /content/drive/MyDrive/TFG_DSM/results/results.csv


,model,best_val_RMSE,test_loss,test_MAE,test_RMSE,test_R2
0,Swin-UNet,5.965727,4.527480,4.976559,8.626991,0.27469
1,Attention-U-Net,3.259811,3.647055,4.061402,7.357884,0.47027


In [ ]:
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

class HRNetRegressor(nn.Module):
    def __init__(self, in_ch=4, backbone="hrnet_w18_small", pretrained=True):
        super().__init__()
        # features_only -> devuelve lista de features; el último suele tener buena resolución
        self.backbone = timm.create_model(
            backbone,
            pretrained=pretrained,
            features_only=True,
            in_chans=in_ch,
        )
        chs = self.backbone.feature_info.channels()
        last_ch = chs[-1]

        # Head de regresión (ligera)
        self.head = nn.Sequential(
            nn.Conv2d(last_ch, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 1, kernel_size=1)
        )

    def forward(self, x):
        H, W = x.shape[-2], x.shape[-1]
        feats = self.backbone(x)

        # timm a veces devuelve NHWC en algunos backbones; lo detectamos:
        f = feats[-1]
        if f.ndim == 4 and f.shape[1] != 128 and f.shape[-1] > 8:
            # heurística simple: si parece NHWC -> pasar a NCHW
            # (B,H,W,C) => (B,C,H,W)
            if f.shape[-1] > 8 and f.shape[1] < 8:
                f = f.permute(0, 3, 1, 2)

        y = self.head(f)

        # subir a tamaño original
        if y.shape[-2:] != (H, W):
            y = F.interpolate(y, size=(H, W), mode="bilinear", align_corners=False)
        return y

In [ ]:
model_hr = HRNetRegressor(in_ch=4, backbone="hrnet_w18_small", pretrained=True).to(device)

x, y = next(iter(train_loader))
x = x.to(device)

with torch.no_grad():
    pred = model_hr(x)

print("pred:", pred.shape, "target:", y.shape)

model.safetensors:   0%|          | 0.00/52.9M [00:00<?, ?B/s]

pred: torch.Size([16, 1, 128, 128]) target: torch.Size([16, 1, 128, 128])


In [ ]:
hr_ckpt = os.path.join(CKPT_DIR, "best_hrnet_w18_small.pth")

history_hr, best_rmse_hr = fit_model_resumable(
    model_hr,
    train_loader,
    val_loader,
    epochs=50,
    lr=1e-4,
    weight_decay=1e-4,
    ckpt_path=hr_ckpt
)

ckpt = torch.load(hr_ckpt, map_location=device)
model_hr.load_state_dict(ckpt["model"])

test_metrics_hr = evaluate_global(model_hr, test_loader)
test_metrics_hr

Reanudando desde checkpoint: /content/drive/MyDrive/TFG_DSM/checkpoints/best_hrnet_w18_small.pth
   -> continúa desde epoch 17
Epoch 017 | lr=7.41e-05 | train_loss=3.2240 | val_RMSE=7.5990 | val_MAE=3.9152 | val_R2=0.1691 | 21.0s
Epoch 018 | lr=7.13e-05 | train_loss=3.2227 | val_RMSE=7.6357 | val_MAE=3.9115 | val_R2=0.1595 | 21.0s
Epoch 019 | lr=6.84e-05 | train_loss=3.2193 | val_RMSE=7.6028 | val_MAE=3.9098 | val_R2=0.1665 | 21.6s
Epoch 020 | lr=6.55e-05 | train_loss=3.2183 | val_RMSE=7.6202 | val_MAE=3.9112 | val_R2=0.1638 | 21.9s
Epoch 021 | lr=6.24e-05 | train_loss=3.2173 | val_RMSE=7.5891 | val_MAE=3.9106 | val_R2=0.1701 | 20.2s
Epoch 022 | lr=5.94e-05 | train_loss=3.2178 | val_RMSE=7.6093 | val_MAE=3.9076 | val_R2=0.1636 | 19.6s
Epoch 023 | lr=5.63e-05 | train_loss=3.2170 | val_RMSE=7.5749 | val_MAE=3.9106 | val_R2=0.1715 | 19.4s
Epoch 024 | lr=5.31e-05 | train_loss=3.2144 | val_RMSE=7.6112 | val_MAE=3.9069 | val_R2=0.1656 | 20.5s
Epoch 025 | lr=5.00e-05 | train_loss=3.2148 | val

{'loss': 5.237060217630296,
 'MAE': 5.702252285821097,
 'RMSE': 9.829071294693719,
 'R2': 0.053116838137308754}

In [ ]:
results.append({
    "model": "HRNet-W18-Small",
    "best_val_RMSE": float(ckpt["best_rmse"]),
    "test_MAE": test_metrics_hr["MAE"],
    "test_RMSE": test_metrics_hr["RMSE"],
    "test_R2": test_metrics_hr["R2"],
})
results

[{'model': 'Swin-UNet',
  'best_val_RMSE': 5.965726858691165,
  'test_loss': 4.527479535057431,
  'test_MAE': 4.9765592983790805,
  'test_RMSE': 8.626991033554077,
  'test_R2': 0.27468970843723844},
 {'model': 'Attention-U-Net',
  'best_val_RMSE': 3.2598106767001904,
  'test_loss': 3.647054677917844,
  'test_MAE': 4.061401821318126,
  'test_RMSE': 7.3578837144942515,
  'test_R2': 0.47026980774743216},
 {'model': 'HRNet-W18-Small',
  'best_val_RMSE': 7.543493797904567,
  'test_loss': 5.237060217630296,
  'test_MAE': 5.702252285821097,
  'test_RMSE': 9.829071294693719,
  'test_R2': 0.053116838137308754}]

In [ ]:

df_results = pd.DataFrame(results)
results_csv_path = os.path.join(RES_DIR, "results.csv")
df_results.to_csv(results_csv_path, index=False)

print("Resultados guardados en:", results_csv_path)
df_results

Resultados guardados en: /content/drive/MyDrive/TFG_DSM/results/results.csv


,model,best_val_RMSE,test_loss,test_MAE,test_RMSE,test_R2
0,Swin-UNet,5.965727,4.527480,4.976559,8.626991,0.274690
1,Attention-U-Net,3.259811,3.647055,4.061402,7.357884,0.470270
2,HRNet-W18-Small,7.543494,5.237060,5.702252,9.829071,0.053117


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    """(conv => BN => ReLU) * 2"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class Down(nn.Module):
    """Downscale with maxpool then double conv"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x):
        return self.conv(self.pool(x))

class Up(nn.Module):
    """Upscale then double conv"""
    def __init__(self, in_ch, skip_ch, out_ch, bilinear=True):
        super().__init__()
        self.bilinear = bilinear
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
            self.conv = DoubleConv(in_ch + skip_ch, out_ch)
        else:
            self.up = nn.ConvTranspose2d(in_ch, in_ch, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_ch=4, base_ch=64, bilinear=True, out_ch=1):
        super().__init__()
        self.inc = DoubleConv(in_ch, base_ch)              # 128
        self.down1 = Down(base_ch, base_ch*2)              # 64
        self.down2 = Down(base_ch*2, base_ch*4)            # 32
        self.down3 = Down(base_ch*4, base_ch*8)            # 16
        self.down4 = Down(base_ch*8, base_ch*16)           # 8

        self.up1 = Up(base_ch*16, base_ch*8,  base_ch*8,  bilinear=bilinear)
        self.up2 = Up(base_ch*8,  base_ch*4,  base_ch*4,  bilinear=bilinear)
        self.up3 = Up(base_ch*4,  base_ch*2,  base_ch*2,  bilinear=bilinear)
        self.up4 = Up(base_ch*2,  base_ch,    base_ch,    bilinear=bilinear)

        self.outc = nn.Conv2d(base_ch, out_ch, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5, x4)
        x = self.up2(x,  x3)
        x = self.up3(x,  x2)
        x = self.up4(x,  x1)

        return self.outc(x)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNetPlusPlus(nn.Module):
    """
    U-Net++ (nested skip connections) para regresión DSM.
    Entrada: (B, in_ch, H, W)
    Salida:  (B, 1, H, W)
    """
    def __init__(self, in_ch=4, base_ch=64, out_ch=1):
        super().__init__()
        self.pool = nn.MaxPool2d(2)

        # Encoder convs (nivel i, j=0)
        self.conv00 = ConvBlock(in_ch, base_ch)
        self.conv10 = ConvBlock(base_ch, base_ch*2)
        self.conv20 = ConvBlock(base_ch*2, base_ch*4)
        self.conv30 = ConvBlock(base_ch*4, base_ch*8)
        self.conv40 = ConvBlock(base_ch*8, base_ch*16)

        # Decoder/nested convs (nivel i, j>0)
        self.conv01 = ConvBlock(base_ch + base_ch*2, base_ch)
        self.conv11 = ConvBlock(base_ch*2 + base_ch*4, base_ch*2)
        self.conv21 = ConvBlock(base_ch*4 + base_ch*8, base_ch*4)
        self.conv31 = ConvBlock(base_ch*8 + base_ch*16, base_ch*8)

        self.conv02 = ConvBlock(base_ch + base_ch + base_ch*2, base_ch)
        self.conv12 = ConvBlock(base_ch*2 + base_ch*2 + base_ch*4, base_ch*2)
        self.conv22 = ConvBlock(base_ch*4 + base_ch*4 + base_ch*8, base_ch*4)

        self.conv03 = ConvBlock(base_ch + base_ch + base_ch + base_ch*2, base_ch)
        self.conv13 = ConvBlock(base_ch*2 + base_ch*2 + base_ch*2 + base_ch*4, base_ch*2)

        self.conv04 = ConvBlock(base_ch + base_ch + base_ch + base_ch + base_ch*2, base_ch)

        self.outc = nn.Conv2d(base_ch, out_ch, kernel_size=1)

    def _up(self, x, ref):
        # upsample x a la resolución de ref
        return F.interpolate(x, size=ref.shape[-2:], mode="bilinear", align_corners=False)

    def forward(self, x):
        # Encoder
        x00 = self.conv00(x)
        x10 = self.conv10(self.pool(x00))
        x20 = self.conv20(self.pool(x10))
        x30 = self.conv30(self.pool(x20))
        x40 = self.conv40(self.pool(x30))

        # Nivel j=1
        x01 = self.conv01(torch.cat([x00, self._up(x10, x00)], dim=1))
        x11 = self.conv11(torch.cat([x10, self._up(x20, x10)], dim=1))
        x21 = self.conv21(torch.cat([x20, self._up(x30, x20)], dim=1))
        x31 = self.conv31(torch.cat([x30, self._up(x40, x30)], dim=1))

        # Nivel j=2
        x02 = self.conv02(torch.cat([x00, x01, self._up(x11, x00)], dim=1))
        x12 = self.conv12(torch.cat([x10, x11, self._up(x21, x10)], dim=1))
        x22 = self.conv22(torch.cat([x20, x21, self._up(x31, x20)], dim=1))

        # Nivel j=3
        x03 = self.conv03(torch.cat([x00, x01, x02, self._up(x12, x00)], dim=1))
        x13 = self.conv13(torch.cat([x10, x11, x12, self._up(x22, x10)], dim=1))

        # Nivel j=4
        x04 = self.conv04(torch.cat([x00, x01, x02, x03, self._up(x13, x00)], dim=1))

        return self.outc(x04)

In [ ]:
# U-Net
model_unet = UNet(in_ch=4, base_ch=64).to(device)
unet_ckpt = os.path.join(CKPT_DIR, "unet_50ep.pth")
hist_unet, best_unet = fit_model_resumable(model_unet, train_loader, val_loader, epochs=50, ckpt_path=unet_ckpt)

# U-Net++
model_unetpp = UNetPlusPlus(in_ch=4, base_ch=64).to(device)
unetpp_ckpt = os.path.join(CKPT_DIR, "unetpp_50ep.pth")
hist_unetpp, best_unetpp = fit_model_resumable(model_unetpp, train_loader, val_loader, epochs=50, ckpt_path=unetpp_ckpt)

NameError: name 'UNet' is not defined

In [ ]:
import os
import pandas as pd
import torch

# Rutas de checkpoints
unet_ckpt = os.path.join(CKPT_DIR, "best_unet.pth")
unetpp_ckpt = os.path.join(CKPT_DIR, "best_unetpp.pth")

# ===== U-NET =====
ckpt = torch.load(unet_ckpt, map_location=device)
model_unet.load_state_dict(ckpt["model"])

test_metrics_unet = evaluate_global(model_unet, test_loader)

results.append({
    "model": "U-Net",
    "best_val_RMSE": float(ckpt["best_rmse"]),
    "test_MAE": test_metrics_unet["MAE"],
    "test_RMSE": test_metrics_unet["RMSE"],
    "test_R2": test_metrics_unet["R2"],
})

# ===== U-NET++ =====
ckpt = torch.load(unetpp_ckpt, map_location=device)
model_unetpp.load_state_dict(ckpt["model"])

test_metrics_unetpp = evaluate(model_unetpp, test_loader)

results.append({
    "model": "U-Net++",
    "best_val_RMSE": float(ckpt["best_rmse"]),

    "test_MAE": test_metrics_unetpp["MAE"],
    "test_RMSE": test_metrics_unetpp["RMSE"],
    "test_R2": test_metrics_unetpp["R2"],
})

# ===== GUARDAR RESULTADOS =====
df_results = pd.DataFrame(results)
results_path = os.path.join(RES_DIR, "results.csv")

df_results.to_csv(results_path, index=False)

print(" Resultados actualizados y guardados en:", results_path)
df_results

In [ ]:
import random
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# ===== ELIGE EL MODELO QUE QUIERES VISUALIZAR =====
model_to_show = model_swin   # o model_unetpp, model_att, model_swin, model_hr
model_name = "Swin-UNet"

# ===== MODO EVALUACIÓN =====
model_to_show.eval()

# ===== ELEGIR UNA MUESTRA ALEATORIA DEL TEST =====
idx = random.randint(0, len(x_test_t) - 1)

x_sample = x_test_t[idx:idx+1].to(device)   # shape: (1, 4, 128, 128)
y_sample = y_test_t[idx:idx+1].to(device)   # shape: (1, 1, 128, 128)

# ===== PREDICCIÓN =====
with torch.no_grad():
    pred = model_to_show(x_sample)
    if pred.shape[-2:] != y_sample.shape[-2:]:
        pred = F.interpolate(pred, size=y_sample.shape[-2:], mode="bilinear", align_corners=False)

# ===== PASAR A NUMPY PARA MOSTRAR =====
x_np = x_sample[0].detach().cpu().numpy()     # (4, 128, 128)
pred_np = pred[0, 0].detach().cpu().numpy()   # (128, 128)
y_np = y_sample[0, 0].detach().cpu().numpy()  # (128, 128)
error = pred_np - y_np

plt.figure(figsize=(5,4))
plt.imshow(error, cmap="coolwarm")
plt.colorbar()
plt.title("Error (Pred - GT)")
plt.show()

# ===== VISUALIZACIÓN =====
fig, axes = plt.subplots(1, 6, figsize=(22, 4))

for c in range(4):
    axes[c].imshow(x_np[c], cmap="gray")
    axes[c].set_title(f"Canal {c+1}")
    axes[c].axis("off")

im_pred = axes[4].imshow(pred_np, cmap="terrain")
axes[4].set_title(f"Salida predicha\n{model_name}")
axes[4].axis("off")
plt.colorbar(im_pred, ax=axes[4], fraction=0.046, pad=0.04)

im_gt = axes[5].imshow(y_np, cmap="terrain")
axes[5].set_title("Ground Truth")
axes[5].axis("off")
plt.colorbar(im_gt, ax=axes[5], fraction=0.046, pad=0.04)

plt.suptitle(f"Muestra aleatoria del test (índice {idx})", fontsize=14)
plt.tight_layout()
plt.show()